## Countries

In [2]:
import pygadm
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
import pycountry

In [ ]:
# the ones that were failing
not_working = [
    "China", 
    "Hong Kong", 
    "India", 
    "Macao", 
    "Pakistan", 
    "Antarctica", 
    "French Southern Territories", 
    "Australia",
    "Brazil",
    "Russia",
    "Bouvet Island",
    "Chile",
    "Greenland"
]
countries = [c for c in pycountry.countries if c.name not in not_working]

gdfs = []
with tqdm(countries, desc="Fetching countries") as pbar:
    for country in pbar:
        pbar.set_postfix(country=country.name)
        gdf = pygadm.Items(admin=country.alpha_3)
        gdfs.append(gdf)

In [ ]:
gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True)).set_crs("EPSG:4326")

In [ ]:
gdf

In [12]:
gdf.to_file("../data/countries.gpkg", driver="GPKG")

In [11]:
gdf = gdf[gdf["NAME_0"] != "Greenland"]

In [ ]:
gdf.shape

In [10]:
gdf = gpd.read_file("../data/countries.gpkg")

In [3]:
gdf.head(3)

,GID_0,NAME_0,geometry
0,ABW,Aruba,"MULTIPOLYGON (((-69.9782 12.4699, -69.9779 12...."
1,AFG,Afghanistan,"MULTIPOLYGON (((63.6155 29.4697, 63.5893 29.47..."
2,AGO,Angola,"MULTIPOLYGON (((19.8989 -17.8767, 19.8908 -17...."


## Rasters

In [17]:
import os


checkpoint_path = "../results/country_confustion_matrix.parquet"

if os.path.exists(checkpoint_path):
    df = pd.read_parquet(checkpoint_path)
    completed = set(df.gid.to_list())
else:
    df = pd.DataFrame(columns=["country", "gid", "pixel_count", "TN", "FP", "FN", "TP"])
    completed = set()

In [8]:
df

,country,gid,pixel_count,TN,FP,FN,TP
0,Grenada,GRD,1715,1185,525,0,5
1,Greece,GRC,794628,689926,100437,30,4235
2,Equatorial Guinea,GNQ,126119,123849,1973,0,297
3,Guinea-Bissau,GNB,161960,161361,340,95,164
4,Gambia,GMB,51298,50218,614,41,425
...,...,...,...,...,...,...,...
74,Åland,ALA,13826,13685,141,0,0
75,Anguilla,AIA,395,96,299,0,0
76,Angola,AGO,5977555,5948467,22567,1057,5464
77,Afghanistan,AFG,3614269,3585649,24612,834,3174


In [13]:
error_path = "../results/country_errors.parquet"

if os.path.exists(error_path):
    error_df = pd.read_parquet(error_path)
    error = set(error_df.gid.to_list())
    completed.update(error)
else:
    error_df = pd.DataFrame(columns=["country", "gid", "error"])

In [14]:
error_df

,country,gid,error
0,Heard Island and McDonald Island,HMD,"7 columns passed, passed data had 4 columns"
1,Fiji,FJI,'coordinates'
2,Christmas Island,CXR,"7 columns passed, passed data had 4 columns"
3,Cocos Islands,CCK,"7 columns passed, passed data had 4 columns"
4,Canada,CAN,Request payload size exceeds the limit: 104857...
5,Botswana,BWA,Manifest from NASA DAAC (https://ladsweb.modap...


In [ ]:
from conflict_monitoring_ntl.satellites import BlackMarblePy, GHSLSurface
from conflict_monitoring_ntl.transform import RasterPipeline
from conflict_monitoring_ntl.utils import binarize_xarray, get_combined_mask, get_non_nan_flat_array
from rasterio.enums import Resampling
import datetime
from sklearn.metrics import confusion_matrix

rasters = [GHSLSurface(), BlackMarblePy(frequency="monthly")]
transformations = [{"reproject_match": {"resampling": Resampling.sum}}, {}]
date = datetime.date(2020, 1, 1)

with tqdm(len(gdf), desc="Calculating confusion matrix") as pbar:
    for i in range(len(gdf)):

        country_gdf = gdf.iloc[[i]]

        country = country_gdf.NAME_0.item()
        gid = country_gdf.GID_0.item()
        pbar.set_postfix(country=country)

        if gid in completed:
            continue 
        
        try:

            pipeline = RasterPipeline(country_gdf, date, rasters, transformations)
            ds = pipeline.run()

            # make sure we compare non-nan areas
            mask = get_combined_mask(ds)
            ds = ds.where(mask)

            ghsl_pop_binary = binarize_xarray(ds.ghsl_surface, 50000)
            y_true = get_non_nan_flat_array(ghsl_pop_binary)

            bm_binary = binarize_xarray(ds.black_marble_radiance_monthly, 1.0)
            y_pred = get_non_nan_flat_array(bm_binary)

            assert y_pred.shape == y_true.shape

            conf_mat = confusion_matrix(y_true, y_pred)

            data = [country, gid, mask.sum().item(), *conf_mat.flatten().tolist()]
            df = pd.concat([pd.DataFrame([data], columns=df.columns), df], ignore_index=True)
            df.to_parquet(checkpoint_path)
        
        except Exception as e:

            data = [country, gid, str(e)]
            error_df = pd.concat([pd.DataFrame([data], columns=error_df.columns), error_df], ignore_index=True)
            error_df.to_parquet(error_path)
            


In [19]:
from copy import deepcopy


df_analysis = deepcopy(df)

In [20]:
df_analysis["f1"]  = 2 * df.TP / (2 * df.TP + df.FP + df.FN)

In [24]:
df_analysis.sort_values(by="f1", ascending=False)

,country,gid,pixel_count,TN,FP,FN,TP,f1
9,Gambia,GMB,51298,50218,614,41,425,0.564784
67,Benin,BEN,547168,539121,3731,1332,2984,0.541021
65,Burkina Faso,BFA,1306636,1299860,3429,835,2512,0.540913
69,Burundi,BDI,126274,125157,429,279,409,0.536042
48,Cameroon,CMR,2193396,2181921,6239,1577,3659,0.483547
...,...,...,...,...,...,...,...,...
21,Falkland Islands,FLK,92856,92817,39,0,0,0.000000
59,Saint-Barthélemy,BLM,107,35,72,0,0,0.000000
77,Andorra,AND,2847,2121,726,0,0,0.000000
79,Åland,ALA,13826,13685,141,0,0,0.000000


In [27]:
from bokeh.plotting import figure, show

fig = figure(title="Scatterplot Example", x_axis_label='x', y_axis_label='y')
fig.scatter(x=df_analysis.pixel_count, y=df_analysis.f1, size=10, color="navy", alpha=0.6)
show(fig)